# Clase 2 — Visión por computador: redes neuronales y convoluciones

Curso **Fundamentos de TensorFlow desde cero** · DIH DATAlife

## Objetivos de la clase

- Entrenar un clasificador de imágenes con una red densa (Fashion MNIST)
- Explicar qué es una convolución y un filtro, y por qué mejoran la visión por computador
- Construir una red neuronal convolucional (CNN) sencilla y comparar su precisión con la red densa
- Interpretar el efecto de las capas de pooling y de distintas capas convolucionales

Este notebook combina teoría (celdas Markdown) y código funcional en TensorFlow 2.x, listo para ejecutar en Google Colab sin instalar nada adicional. Consulta los capítulos 3 y 4 del manual del curso para la explicación completa.

## Parte 1 · Una red densa para "ver" imágenes

Un ordenador no ve ropa: ve una cuadrícula de números. Una imagen en escala de grises de 28x28 píxeles es una matriz de 28x28 valores entre 0 y 255, uno por píxel. La visión por computador con Machine Learning (ML) consiste en enseñarle al modelo miles de ejemplos etiquetados para que aprenda por sí mismo qué combinaciones de esos números corresponden a cada clase, en vez de programar reglas a mano.

Vamos a entrenar una primera red neuronal densa con el dataset **Fashion MNIST**: 70.000 imágenes de prendas de ropa (60.000 de entrenamiento y 10.000 de prueba), en escala de grises, de 28x28 píxeles, repartidas en diez categorías.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print('Versión de TensorFlow:', tf.__version__)

In [ ]:
# Cargamos Fashion MNIST directamente desde la API de datasets de Keras
fashion_mnist = tf.keras.datasets.fashion_mnist
(training_images, training_labels), (test_images, test_labels) = fashion_mnist.load_data()

class_names = ['Camiseta', 'Pantalón', 'Jersey', 'Vestido', 'Abrigo',
               'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']

print('Forma de las imágenes de entrenamiento:', training_images.shape)
print('Número de etiquetas de entrenamiento:', len(training_labels))

plt.figure()
plt.imshow(training_images[0], cmap='gray')
plt.title(f'Etiqueta: {training_labels[0]} ({class_names[training_labels[0]]})')
plt.colorbar()
plt.grid(False)
plt.show()

### Normalización de los datos

Cada píxel vale entre 0 y 255. Las redes entrenan mejor cuando los valores de entrada están en un rango pequeño y homogéneo, así que **normalizamos** dividiendo entre 255. Es importante aplicar exactamente la misma transformación al conjunto de entrenamiento y al de prueba.

In [ ]:
training_images_norm = training_images / 255.0
test_images_norm = test_images / 255.0

### Capas, activaciones y softmax

La red densa tiene tres piezas:

- **Flatten**: convierte la imagen 28x28 en un vector de 784 valores, sin aprender nada
- **Dense(128, activation='relu')**: una capa oculta de 128 neuronas. La activación **ReLU** (Rectified Linear Unit, Unidad Lineal Rectificada) deja pasar los valores positivos tal cual y convierte en cero los negativos
- **Dense(10, activation='softmax')**: la capa de salida, con una neurona por clase. **Softmax** convierte los valores de salida en probabilidades que suman 1, así que la clase con la probabilidad más alta es la predicción del modelo

Compilamos el modelo con el optimizador `adam`, la función de pérdida `sparse_categorical_crossentropy` (porque las etiquetas son números enteros) y la métrica `accuracy`.

In [ ]:
modelo_denso = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

modelo_denso.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

modelo_denso.fit(training_images_norm, training_labels, epochs=5)

In [ ]:
loss_denso, accuracy_denso = modelo_denso.evaluate(test_images_norm, test_labels)
print(f'\nExactitud del modelo denso en el conjunto de prueba: {accuracy_denso:.4f}')

predicciones_denso = modelo_denso.predict(test_images_norm)
print('Probabilidades para la primera imagen de prueba:', predicciones_denso[0])
print('Clase predicha:', np.argmax(predicciones_denso[0]), '- Clase real:', test_labels[0])

### Los límites de la red densa

Esta red densa aprende directamente de los píxeles en bruto. Funciona bien con Fashion MNIST porque todas las imágenes están centradas, en escala de grises y con el mismo tamaño exacto, pero esa misma dependencia es su límite: no tiene ningún concepto de "borde" o "forma", solo memoriza patrones de píxeles. Con una foto real, con fondo o con la prenda desplazada, esta red falla.

Para superar esa limitación necesitamos que el modelo aprenda primero a detectar **rasgos visuales** (bordes, curvas, texturas) antes de clasificar. A eso se dedica la segunda parte del notebook: las convoluciones.

## Parte 2 · Qué es una convolución

Una **convolución** desliza un filtro (una pequeña matriz de números, también llamado kernel) por toda la imagen. En cada posición, el filtro combina el píxel central con sus vecinos: multiplica cada píxel por el valor del filtro que le corresponde y suma los resultados. Ese es el nuevo valor del píxel en la imagen transformada.

Vamos a comprobarlo con una imagen de ejemplo (la clásica "ascent" que trae `scipy`) y un filtro que resalta bordes horizontales.

In [ ]:
try:
    from scipy import datasets as scipy_datasets
    imagen_original = scipy_datasets.ascent()
except ImportError:
    from scipy import misc
    imagen_original = misc.ascent()

plt.gray()
plt.grid(False)
plt.axis('off')
plt.imshow(imagen_original)
plt.title('Imagen original')
plt.show()

print('Forma de la imagen:', imagen_original.shape)

In [ ]:
def aplicar_filtro(imagen, filtro, peso=1.0):
    """Aplica manualmente un filtro de convolución 3x3 a una imagen en escala de grises."""
    imagen_filtrada = np.copy(imagen)
    size_x, size_y = imagen.shape

    for x in range(1, size_x - 1):
        for y in range(1, size_y - 1):
            convolucion = 0.0
            for fx in range(3):
                for fy in range(3):
                    convolucion += imagen[x + fx - 1, y + fy - 1] * filtro[fx][fy]
            convolucion *= peso
            convolucion = max(0, min(255, convolucion))
            imagen_filtrada[x, y] = convolucion

    return imagen_filtrada

# Filtro que resalta bordes horizontales
filtro_horizontal = [[-1, -2, -1],
                      [ 0,  0,  0],
                      [ 1,  2,  1]]

imagen_convolucionada = aplicar_filtro(imagen_original, filtro_horizontal)

plt.gray()
plt.grid(False)
plt.axis('off')
plt.imshow(imagen_convolucionada)
plt.title('Después de aplicar el filtro (bordes horizontales)')
plt.show()

Prueba a cambiar `filtro_horizontal` por `[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]` (bordes verticales) y vuelve a ejecutar la celda anterior: verás que se resaltan las líneas verticales de la imagen en vez de las horizontales. El filtro determina qué rasgo se hace visible, pero en una CNN real esos números no los eliges tú: la red los aprende sola durante el entrenamiento, ajustándolos con retropropagación igual que ajusta los pesos de una capa densa.

### Pooling: resumir sin perder lo importante

El **pooling** reduce el tamaño de la imagen conservando lo esencial de lo que el filtro ya detectó. La variante más común es **max pooling**: se recorre la imagen en bloques (por ejemplo, de 2x2) y de cada bloque solo se conserva el valor más alto.

In [ ]:
def max_pooling(imagen, tamano=2):
    """Aplica max pooling de tamano x tamano sobre una imagen en escala de grises."""
    size_x, size_y = imagen.shape
    nueva_x, nueva_y = size_x // tamano, size_y // tamano
    imagen_reducida = np.zeros((nueva_x, nueva_y))

    for x in range(0, size_x - 1, tamano):
        for y in range(0, size_y - 1, tamano):
            bloque = imagen[x:x + tamano, y:y + tamano]
            imagen_reducida[x // tamano, y // tamano] = np.max(bloque)

    return imagen_reducida

imagen_pooled = max_pooling(imagen_convolucionada, tamano=2)

print('Tamaño antes del pooling:', imagen_convolucionada.shape)
print('Tamaño después del pooling:', imagen_pooled.shape)

plt.gray()
plt.grid(False)
plt.axis('off')
plt.imshow(imagen_pooled)
plt.title('Después de max pooling 2x2')
plt.show()

La imagen resultante ocupa la cuarta parte de píxeles que la original, pero los rasgos que el filtro había resaltado siguen siendo reconocibles. Esa es la doble ventaja del pooling: menos cálculo para las siguientes capas y más robustez frente a pequeños desplazamientos del objeto en la imagen.

## Parte 3 · Construir una CNN completa en Keras

En Keras no aplicamos las convoluciones y el pooling a mano: usamos las capas `Conv2D` y `MaxPooling2D`. Una CNN típica encadena varios bloques de convolución + pooling para extraer características, y termina con `Flatten` y capas `Dense` para clasificar, igual que la red densa de la Parte 1.

`Conv2D` espera imágenes con un canal explícito, así que necesitamos añadir esa dimensión a Fashion MNIST: de `(28, 28)` a `(28, 28, 1)`.

In [ ]:
training_images_cnn = training_images_norm.reshape(-1, 28, 28, 1)
test_images_cnn = test_images_norm.reshape(-1, 28, 28, 1)

modelo_cnn = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

modelo_cnn.summary()

In [ ]:
modelo_cnn.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

modelo_cnn.fit(training_images_cnn, training_labels, epochs=5)

In [ ]:
loss_cnn, accuracy_cnn = modelo_cnn.evaluate(test_images_cnn, test_labels)

print(f'Exactitud red densa (Parte 1): {accuracy_denso:.4f}')
print(f'Exactitud CNN       (Parte 3): {accuracy_cnn:.4f}')

Con el mismo número de épocas, la CNN suele igualar o mejorar la exactitud de la red densa, y esa ventaja se hace mucho más notable cuando las imágenes son más grandes, más complejas o el objeto no está perfectamente centrado, justo el escenario donde la red densa fallaba.

## Parte 4 · Un clasificador de imágenes de principio a fin

Un clasificador completo no termina en `model.fit`: incluye interpretar la predicción final. Como la última capa usa softmax, `model.predict` devuelve una probabilidad por clase, y `np.argmax` te da la clase más probable. Vamos a definir dos funciones para visualizar, para una imagen de prueba, la predicción de la CNN junto a su confianza en cada una de las diez clases.

In [ ]:
def plot_imagen(indice, predicciones, etiquetas_reales, imagenes):
    prediccion = predicciones[indice]
    etiqueta_real = etiquetas_reales[indice]
    imagen = imagenes[indice].reshape(28, 28)

    clase_predicha = np.argmax(prediccion)
    color = 'blue' if clase_predicha == etiqueta_real else 'red'

    plt.grid(False)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(imagen, cmap=plt.cm.binary)
    plt.xlabel('{} {:2.0f}% (real: {})'.format(
        class_names[clase_predicha],
        100 * np.max(prediccion),
        class_names[etiqueta_real]),
        color=color)


def plot_probabilidades(indice, predicciones, etiquetas_reales):
    prediccion = predicciones[indice]
    etiqueta_real = etiquetas_reales[indice]

    plt.grid(False)
    plt.xticks(range(10))
    plt.yticks([])
    barras = plt.bar(range(10), prediccion, color='#777777')
    plt.ylim([0, 1])

    clase_predicha = np.argmax(prediccion)
    barras[clase_predicha].set_color('red')
    barras[etiqueta_real].set_color('blue')


predicciones_cnn = modelo_cnn.predict(test_images_cnn)

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plot_imagen(0, predicciones_cnn, test_labels, test_images_cnn)
plt.subplot(1, 2, 2)
plot_probabilidades(0, predicciones_cnn, test_labels)
plt.show()

In [ ]:
# Predicción sobre una única imagen "nueva": recuerda que Keras siempre espera un lote (batch)
imagen_nueva = test_images_cnn[1]
lote = np.expand_dims(imagen_nueva, axis=0)   # forma (1, 28, 28, 1)

prediccion_unica = modelo_cnn.predict(lote)
clase_predicha = np.argmax(prediccion_unica[0])

print(f'Clase predicha: {class_names[clase_predicha]} ({clase_predicha})')
print(f'Clase real: {class_names[test_labels[1]]} ({test_labels[1]})')
print(f'Confianza: {np.max(prediccion_unica[0]):.2%}')

## Resumen y siguientes pasos

- Una red densa clasifica imágenes aprendiendo directamente de los píxeles en bruto, pero solo funciona bien con imágenes centradas y del mismo tamaño
- Una convolución filtra la imagen para resaltar rasgos (bordes, texturas); los valores del filtro se aprenden durante el entrenamiento, no se diseñan a mano
- El pooling reduce el tamaño de la imagen conservando los rasgos más importantes, aligerando el cálculo y dando robustez frente a desplazamientos
- Una CNN combina bloques `Conv2D` + `MaxPooling2D` para extraer características y termina con `Flatten` + `Dense` para clasificar, igual que la red densa
- Interpretar la salida de un clasificador significa leer las probabilidades de softmax con `np.argmax`, no solo mirar si `accuracy` subió

Ahora abre el notebook de práctica de esta clase (`clase02_practica_vision_por_computador_cnn.ipynb`) para modificar tú mismo la arquitectura de la CNN y medir el efecto en la precisión.